

## Project on Responsible Data Science

## 1. Classifier

In this section we implement a Xgboost classifier and we also binarized the age.

#### Importation of the librairies needed

In [13]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder


### Load and preprocess the dataset

In [14]:
# 1. Define Paths and Columns
# Column names (based on UCI Adult documentation since the file has no header)
columns = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status", 
           "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss", 
           "hours-per-week", "native-country", "income"]

# Relative path
file_path = os.path.join("..", "data", "raw", "adult.data")

# 2. Load Data
try:
    # sep=',\s*' handles spaces after commas (e.g., ", White")
    df = pd.read_csv(file_path, names=columns, sep=r',\s*', engine='python')
    print(f"✅ Dataset loaded successfully! Shape: {df.shape}")
except FileNotFoundError:
    print(f"❌ Error: File not found at: {os.path.abspath(file_path)}")
    print("Please verify that your file is located in 'data/raw/adult.data' at the project root.")
    # Fallback: Try loading without ".." if running from root
    try: 
        file_path_alt = os.path.join("data", "raw", "adult.data")
        df = pd.read_csv(file_path_alt, names=columns, sep=r',\s*', engine='python')
        print(f"✅ Dataset loaded with alternative path! Shape: {df.shape}")
    except:
        print("❌ Loading failed. Please check your folder structure.")

# 3. Cleaning
# Replace '?' with NaN and drop missing rows
df.replace('?', np.nan, inplace=True)
df.dropna(inplace=True)

# 4. Binarize Age (Required by Project Step 1)
# Calculate median age
median_age = df['age'].median()
print(f"ℹ️ Median Age used for cut: {median_age}")

# Create binary column (0 = <= median, 1 = > median)
df['age_binary'] = df['age'].apply(lambda x: 0 if x <= median_age else 1)

# Drop original age column (to avoid leaking info or redundancy)
df = df.drop('age', axis=1)

# 5. Encoding
# Encode target (income): <=50K -> 0, >50K -> 1
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

# Encode categorical variables (One-Hot Encoding)
categorical_cols = df.select_dtypes(include=['object']).columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("\nDataframe Head (Processed):")
print(df.head())

✅ Dataset loaded successfully! Shape: (32561, 15)
ℹ️ Median Age used for cut: 37.0

Dataframe Head (Processed):
   fnlwgt  education-num  capital-gain  capital-loss  hours-per-week  income  \
0   77516             13          2174             0              40       0   
1   83311             13             0             0              13       0   
2  215646              9             0             0              40       0   
3  234721              7             0             0              40       0   
4  338409             13             0             0              40       0   

   age_binary  workclass_Local-gov  workclass_Private  workclass_Self-emp-inc  \
0           1                False              False                   False   
1           1                False              False                   False   
2           1                False               True                   False   
3           1                False               True                   False   
4 

### Split and Train Data

In [10]:
# Divide into features (X) and labels (y)
X = df.drop('income', axis=1)
y = df['income']

# The project requires Train / Validation / Test splits
# Split 1: Keep 60% for Train, the rest (40%) goes to Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42
)

# Split 2: Split Temp in two (50/50) -> 20% Val, 20% Test overall
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Train size: {len(X_train)} (60%)")
print(f"Validation size: {len(X_val)} (20%)")
print(f"Test size: {len(X_test)} (20%)")

# Train an XGBoost classifier
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

Train size: 18097 (60%)
Validation size: 6032 (20%)
Test size: 6033 (20%)


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


### Test the Model

In [12]:
# Predictions on Test Set
y_pred = model.predict(X_test)

# Evaluate performance
print("MODEL PERFORMANCE ON TEST SET")
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

MODEL PERFORMANCE ON TEST SET
Accuracy: 0.8677

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.93      0.91      4478
           1       0.78      0.68      0.73      1555

    accuracy                           0.87      6033
   macro avg       0.84      0.81      0.82      6033
weighted avg       0.86      0.87      0.86      6033

